Import des librairies

In [5]:
# Cell 0: Load Config & Init Spark
import yaml
import pathlib
import datetime
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Load Configuration
config_path = "de1_project_config.yml"
with open(config_path) as f:
    CFG = yaml.safe_load(f)

# Initialize Spark Session (Local Mode)
spark = SparkSession.builder \
    .appName(CFG["project_name"]) \
    .config("spark.master", "local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"✅ Spark Version: {spark.version}")
print(f"✅ Config Loaded. Raw Path: {CFG['paths']['raw_landing']}")
print(f"✅ App Name: {spark.sparkContext.appName}")

✅ Spark Version: 4.0.0-preview1
✅ Config Loaded. Raw Path: /home/Zettahorizon/projects/de1_wikipedia/data/raw_landing
✅ App Name: DE1_Wikipedia_Lakehouse


Bronze Layer (ingestion)

In [9]:
# FUSION SETUP + BRONZE
import yaml
from pyspark.sql import SparkSession

# 1. Chargement Config
with open("de1_project_config.yml") as f:
    CFG = yaml.safe_load(f)

# 2. Init Spark
spark = SparkSession.builder \
    .appName(CFG["project_name"]) \
    .config("spark.master", "local[*]") \
    .getOrCreate()

# 3. Bronze Layer
raw_path = CFG["paths"]["raw_landing"]
bronze_path = CFG["paths"]["bronze"]

df_raw = (spark.read
    .option("delimiter", "\t")
    .option("header", "false")
    .csv(f"{raw_path}/*.tsv.gz")
)

df_raw = df_raw.toDF("prev", "curr", "type", "n")
df_raw.write.mode("overwrite").parquet(bronze_path)

print(f"✅ Bronze terminé. Lignes : {spark.read.parquet(bronze_path).count():,}")

{"ts":"2026-01-03T17:06:14.001Z","level":"WARN","msg":"Assume no metadata directory. Error while looking for metadata directory in the path: /home/zettahorizon/projects/de1_wikipedia/data/raw_landing/*.tsv.gz.","exception":{"class":"java.io.FileNotFoundException","msg":"File /home/zettahorizon/projects/de1_wikipedia/data/raw_landing/*.tsv.gz does not exist","stacktrace":[{"class":"org.apache.hadoop.fs.RawLocalFileSystem","method":"deprecatedGetFileStatus","file":"RawLocalFileSystem.java","line":915},{"class":"org.apache.hadoop.fs.RawLocalFileSystem","method":"getFileLinkStatusInternal","file":"RawLocalFileSystem.java","line":1236},{"class":"org.apache.hadoop.fs.RawLocalFileSystem","method":"getFileStatus","file":"RawLocalFileSystem.java","line":905},{"class":"org.apache.hadoop.fs.FilterFileSystem","method":"getFileStatus","file":"FilterFileSystem.java","line":462},{"class":"org.apache.spark.sql.execution.streaming.FileStreamSink$","method":"hasMetadata","file":"FileStreamSink.scala","l

✅ Bronze terminé. Lignes : 35,605,767


{"ts":"2026-01-03T17:07:18.004Z","level":"INFO","msg":"Getting 20 (1200.0 B) non-empty blocks including 20 (1200.0 B) local and 0 (0.0 B) host-local and 0 (0.0 B) push-merged-local and 0 (0.0 B) remote blocks","context":{"task_name":"task 0.0 in stage 7.0 (TID 25)"},"logger":"ShuffleBlockFetcherIterator"}
{"ts":"2026-01-03T17:07:18.007Z","level":"INFO","msg":"Started 0 remote fetches in 20 ms","context":{"task_name":"task 0.0 in stage 7.0 (TID 25)"},"logger":"ShuffleBlockFetcherIterator"}
{"ts":"2026-01-03T17:07:18.033Z","level":"INFO","msg":"Code generated in 16.820874 ms","context":{"task_name":"task 0.0 in stage 7.0 (TID 25)"},"logger":"CodeGenerator"}
{"ts":"2026-01-03T17:07:18.053Z","level":"INFO","msg":"Finished task 0.0 in stage 7.0 (TID 25). 3895 bytes result sent to driver","context":{"task_name":"task 0.0 in stage 7.0 (TID 25)"},"logger":"Executor"}
{"ts":"2026-01-03T17:07:18.055Z","level":"INFO","msg":"Finished task 0.0 in stage 7.0 (TID 25) in 102 ms on 10.255.255.254 (exec

In [11]:
# Sauvegarde du plan pour le passage Raw -> Bronze
plan_bronze = df_raw._jdf.queryExecution().executedPlan().toString()
with open(f"{CFG['paths']['proof']}/baseline_bronze_plan.txt", "w") as f:
    f.write(plan_bronze)
print("✅ Preuve Bronze sauvegardée.")

✅ Preuve Bronze sauvegardée.


{"ts":"2026-01-03T17:23:23.591Z","level":"INFO","msg":"Pushed Filters: ","logger":"FileSourceStrategy"}
{"ts":"2026-01-03T17:23:23.592Z","level":"INFO","msg":"Post-Scan Filters: ","logger":"FileSourceStrategy"}


Silver Nettoyage

In [10]:
from pyspark.sql import functions as F

print("🥈 Transition vers la couche Silver...")

# 1. Lecture de la couche Bronze
df_bronze = spark.read.parquet(CFG["paths"]["bronze"])

# 2. Nettoyage et Typage
# - On convertit 'n' en Integer
# - On peut filtrer les clics négatifs ou nuls si nécessaire
df_silver = (df_bronze
    .withColumn("n", F.col("n").cast("integer"))
    .filter(F.col("n") > 0)
    .dropna(subset=["curr", "n"]) # On garde les lignes essentielles
)

# 3. Écriture de la couche Silver
silver_path = CFG["paths"]["silver"]
df_silver.write.mode("overwrite").parquet(silver_path)

# 4. Métriques de contrôle
count_bronze = df_bronze.count()
count_silver = df_silver.count()
dropped = count_bronze - count_silver

print(f"✅ Silver Layer écrite à : {silver_path}")
print(f"📊 Rows in Bronze: {count_bronze:,}")
print(f"📊 Rows in Silver: {count_silver:,}")
print(f"🗑️ Rows dropped (cleaning): {dropped:,}")

🥈 Transition vers la couche Silver...


{"ts":"2026-01-03T17:21:29.493Z","level":"INFO","msg":"It took 4 ms to list leaf files for 1 paths.","logger":"InMemoryFileIndex"}
{"ts":"2026-01-03T17:21:29.547Z","level":"INFO","msg":"Starting job: parquet at NativeMethodAccessorImpl.java:0","logger":"SparkContext"}
{"ts":"2026-01-03T17:21:29.550Z","level":"INFO","msg":"Got job 7 (parquet at NativeMethodAccessorImpl.java:0) with 1 output partitions","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:21:29.550Z","level":"INFO","msg":"Final stage: ResultStage 8 (parquet at NativeMethodAccessorImpl.java:0)","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:21:29.550Z","level":"INFO","msg":"Parents of final stage: List()","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:21:29.551Z","level":"INFO","msg":"Missing parents: List()","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:21:29.552Z","level":"INFO","msg":"Submitting ResultStage 8 (MapPartitionsRDD[38] at parquet at NativeMethodAccessorImpl.java:0), which has no missing parents","logger":"DAGSchedul

✅ Silver Layer écrite à : /home/zettahorizon/projects/de1_wikipedia/data/silver
📊 Rows in Bronze: 35,605,767
📊 Rows in Silver: 35,605,763
🗑️ Rows dropped (cleaning): 4


{"ts":"2026-01-03T17:21:40.306Z","level":"INFO","msg":"Removed broadcast_15_piece0 on 10.255.255.254:36541 in memory (size: 39.0 KiB, free: 2.2 GiB)","logger":"BlockManagerInfo"}
{"ts":"2026-01-03T17:21:40.316Z","level":"INFO","msg":"Finished task 2.0 in stage 13.0 (TID 70). 2233 bytes result sent to driver","context":{"task_name":"task 2.0 in stage 13.0 (TID 70)"},"logger":"Executor"}
{"ts":"2026-01-03T17:21:40.317Z","level":"INFO","msg":"Finished task 6.0 in stage 13.0 (TID 74). 2233 bytes result sent to driver","context":{"task_name":"task 6.0 in stage 13.0 (TID 74)"},"logger":"Executor"}
{"ts":"2026-01-03T17:21:40.319Z","level":"INFO","msg":"Finished task 6.0 in stage 13.0 (TID 74) in 1001 ms on 10.255.255.254 (executor driver) (16/20)","logger":"TaskSetManager"}
{"ts":"2026-01-03T17:21:40.319Z","level":"INFO","msg":"Finished task 2.0 in stage 13.0 (TID 70) in 1006 ms on 10.255.255.254 (executor driver) (17/20)","logger":"TaskSetManager"}
{"ts":"2026-01-03T17:21:40.337Z","level":"I

In [12]:
import datetime as _dt
import pathlib

# Créer le dossier de preuves s'il n'existe pas
proof_path = CFG["paths"]["proof"]
pathlib.Path(proof_path).mkdir(parents=True, exist_ok=True)

# Capturer le plan physique de la transformation Silver
plan = df_silver._jdf.queryExecution().executedPlan().toString()

with open(f"{proof_path}/silver_transformation_plan.txt", "w") as f:
    f.write(f"Timestamp: {_dt.datetime.now()}\n")
    f.write("Plan physique de la couche Silver (Cleaning & Casting):\n")
    f.write(plan)

print(f"📄 Plan physique sauvegardé dans : {proof_path}/silver_transformation_plan.txt")

📄 Plan physique sauvegardé dans : /home/zettahorizon/projects/de1_wikipedia/proof/silver_transformation_plan.txt


{"ts":"2026-01-03T17:23:34.319Z","level":"INFO","msg":"Pushed Filters: IsNotNull(n)","logger":"FileSourceStrategy"}
{"ts":"2026-01-03T17:23:34.322Z","level":"INFO","msg":"Post-Scan Filters: isnotnull(n#104),(cast(n#104 as int) > 0),atleastnnonnulls(2, curr#102, cast(n#104 as int))","logger":"FileSourceStrategy"}


Comptage du temps

In [13]:
import datetime
import os

metrics_path = CFG["paths"]["metrics"]

# Récupération des nombres de lignes (pour le log)
rows_bronze = spark.read.parquet(CFG["paths"]["bronze"]).count()
rows_silver = spark.read.parquet(CFG["paths"]["silver"]).count()

# Préparation des lignes à écrire
# Format: run_id, timestamp, layer, operation, duration_sec, input_rows, output_rows, notes
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

lines_to_append = [
    f"run_1,{timestamp},bronze,ingest_csv_to_parquet,64.1,0,{rows_bronze},Raw TSV to Bronze",
    f"run_1,{timestamp},silver,clean_and_cast,10.9,{rows_bronze},{rows_silver},Cast n to Int & DropNulls"
]

# Écriture dans le fichier (mode 'a' pour append)
with open(metrics_path, "a") as f:
    for line in lines_to_append:
        f.write(line + "\n")

print(f"✅ Métriques enregistrées dans : {metrics_path}")
print("   - Bronze: 64.1s")
print("   - Silver: 10.9s")

{"ts":"2026-01-03T17:28:01.759Z","level":"INFO","msg":"It took 8 ms to list leaf files for 1 paths.","logger":"InMemoryFileIndex"}
{"ts":"2026-01-03T17:28:01.814Z","level":"INFO","msg":"Starting job: parquet at NativeMethodAccessorImpl.java:0","logger":"SparkContext"}
{"ts":"2026-01-03T17:28:01.816Z","level":"INFO","msg":"Got job 13 (parquet at NativeMethodAccessorImpl.java:0) with 1 output partitions","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:28:01.816Z","level":"INFO","msg":"Final stage: ResultStage 16 (parquet at NativeMethodAccessorImpl.java:0)","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:28:01.816Z","level":"INFO","msg":"Parents of final stage: List()","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:28:01.817Z","level":"INFO","msg":"Missing parents: List()","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:28:01.818Z","level":"INFO","msg":"Submitting ResultStage 16 (MapPartitionsRDD[58] at parquet at NativeMethodAccessorImpl.java:0), which has no missing parents","logger":"DAGSche

✅ Métriques enregistrées dans : /home/zettahorizon/projects/de1_wikipedia/project_metrics_log.csv
   - Bronze: 64.1s
   - Silver: 10.9s


{"ts":"2026-01-03T17:28:02.619Z","level":"INFO","msg":"Starting job: count at NativeMethodAccessorImpl.java:0","logger":"SparkContext"}
{"ts":"2026-01-03T17:28:02.621Z","level":"INFO","msg":"Got job 18 (count at NativeMethodAccessorImpl.java:0) with 1 output partitions","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:28:02.621Z","level":"INFO","msg":"Final stage: ResultStage 23 (count at NativeMethodAccessorImpl.java:0)","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:28:02.621Z","level":"INFO","msg":"Parents of final stage: List(ShuffleMapStage 22)","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:28:02.621Z","level":"INFO","msg":"Missing parents: List()","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:28:02.621Z","level":"INFO","msg":"Submitting ResultStage 23 (MapPartitionsRDD[74] at count at NativeMethodAccessorImpl.java:0), which has no missing parents","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:28:02.626Z","level":"INFO","msg":"Block broadcast_30 stored as values in memory (estimated s

{"ts":"2026-01-03T17:29:03.628Z","level":"INFO","msg":"Removed broadcast_22_piece0 on 10.255.255.254:36541 in memory (size: 6.5 KiB, free: 2.2 GiB)","logger":"BlockManagerInfo"}
{"ts":"2026-01-03T17:29:03.638Z","level":"INFO","msg":"Removed broadcast_29_piece0 on 10.255.255.254:36541 in memory (size: 8.3 KiB, free: 2.2 GiB)","logger":"BlockManagerInfo"}
{"ts":"2026-01-03T17:29:03.647Z","level":"INFO","msg":"Removed broadcast_27_piece0 on 10.255.255.254:36541 in memory (size: 41.0 KiB, free: 2.2 GiB)","logger":"BlockManagerInfo"}
{"ts":"2026-01-03T17:29:03.656Z","level":"INFO","msg":"Removed broadcast_30_piece0 on 10.255.255.254:36541 in memory (size: 6.5 KiB, free: 2.2 GiB)","logger":"BlockManagerInfo"}
{"ts":"2026-01-03T17:29:03.667Z","level":"INFO","msg":"Removed broadcast_28_piece0 on 10.255.255.254:36541 in memory (size: 39.0 KiB, free: 2.2 GiB)","logger":"BlockManagerInfo"}
{"ts":"2026-01-03T17:29:03.678Z","level":"INFO","msg":"Removed broadcast_21_piece0 on 10.255.255.254:36541 i

Gold Données prêtes à l'emploi

In [14]:
import time

print("🥇 Lancement de la Gold Layer - Q1 Baseline (Agrégation)...")

# 1. Lecture de la Silver
df_silver = spark.read.parquet(CFG["paths"]["silver"])

# 2. Définition de la requête Q1 (Sans optimisation)
# On groupe par article ('curr') et on somme les clics ('n')
gold_q1 = (df_silver
           .groupBy("curr")
           .agg(F.sum("n").alias("total_clicks"))
           .orderBy(F.col("total_clicks").desc()) # On trie pour voir les champions
          )

# 3. Mesure du temps d'écriture (Action)
start_time = time.time()

# Attention : On écrit SANS partitionning pour la baseline
gold_path_q1 = f"{CFG['paths']['gold']}/q1_baseline"
gold_q1.write.mode("overwrite").parquet(gold_path_q1)

end_time = time.time()
duration = round(end_time - start_time, 2)

# 4. Sauvegarde du Plan Physique (Preuve essentielle)
plan = gold_q1._jdf.queryExecution().executedPlan().toString()
with open(f"{CFG['paths']['proof']}/baseline_q1_plan.txt", "w") as f:
    f.write(f"Timestamp: {datetime.datetime.now()}\n")
    f.write(f"Duration: {duration}s\n")
    f.write("Plan Baseline Q1 (Scan Silver -> Shuffle -> Write Gold):\n")
    f.write(plan)

print("-" * 30)
print(f"✅ Gold Q1 terminée en : {duration} secondes")
print(f"📂 Données écrites dans : {gold_path_q1}")
print(f"📄 Plan sauvegardé : baseline_q1_plan.txt")

# Petit aperçu des résultats
print("🏆 Top 5 Articles du mois :")
spark.read.parquet(gold_path_q1).show(5, truncate=False)

🥇 Lancement de la Gold Layer - Q1 Baseline (Agrégation)...


{"ts":"2026-01-03T17:29:09.779Z","level":"INFO","msg":"It took 6 ms to list leaf files for 1 paths.","logger":"InMemoryFileIndex"}
{"ts":"2026-01-03T17:29:09.837Z","level":"INFO","msg":"Starting job: parquet at NativeMethodAccessorImpl.java:0","logger":"SparkContext"}
{"ts":"2026-01-03T17:29:09.840Z","level":"INFO","msg":"Got job 19 (parquet at NativeMethodAccessorImpl.java:0) with 1 output partitions","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:29:09.840Z","level":"INFO","msg":"Final stage: ResultStage 24 (parquet at NativeMethodAccessorImpl.java:0)","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:29:09.840Z","level":"INFO","msg":"Parents of final stage: List()","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:29:09.841Z","level":"INFO","msg":"Missing parents: List()","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:29:09.842Z","level":"INFO","msg":"Submitting ResultStage 24 (MapPartitionsRDD[76] at parquet at NativeMethodAccessorImpl.java:0), which has no missing parents","logger":"DAGSche

------------------------------
✅ Gold Q1 terminée en : 9.44 secondes
📂 Données écrites dans : /home/zettahorizon/projects/de1_wikipedia/data/gold/q1_baseline
📄 Plan sauvegardé : baseline_q1_plan.txt
🏆 Top 5 Articles du mois :


{"ts":"2026-01-03T17:29:19.866Z","level":"INFO","msg":"Finished task 0.0 in stage 33.0 (TID 218). 2129 bytes result sent to driver","context":{"task_name":"task 0.0 in stage 33.0 (TID 218)"},"logger":"Executor"}
{"ts":"2026-01-03T17:29:19.868Z","level":"INFO","msg":"Finished task 0.0 in stage 33.0 (TID 218) in 50 ms on 10.255.255.254 (executor driver) (1/1)","logger":"TaskSetManager"}
{"ts":"2026-01-03T17:29:19.868Z","level":"INFO","msg":"Removed TaskSet 33.0, whose tasks have all completed, from pool ","logger":"TaskSchedulerImpl"}
{"ts":"2026-01-03T17:29:19.868Z","level":"INFO","msg":"ResultStage 33 (parquet at NativeMethodAccessorImpl.java:0) finished in 60 ms","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:29:19.869Z","level":"INFO","msg":"Job 24 is finished. Cancelling potential speculative or zombie tasks for this job","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:29:19.870Z","level":"INFO","msg":"Canceling stage 33","logger":"TaskSchedulerImpl"}
{"ts":"2026-01-03T17:29:19.870Z","

+----------------------------+------------+
|curr                        |total_clicks|
+----------------------------+------------+
|Shielmartin_Hill            |23          |
|Christian_Montanari         |23          |
|The_Weeding_of_Covent_Garden|23          |
|People's_Park_(Shenzhen)    |23          |
|Alex_Shvartsman             |23          |
+----------------------------+------------+
only showing top 5 rows



In [15]:
print("⚡ Lancement de la Gold Layer - Q1 OPTIMIZED...")

# 1. APPLICATION DES OPTIMISATIONS
# On réduit le nombre de partitions de shuffle (Default = 200 -> Optimized = 12)
# Cela réduit l'overhead du CPU sur une machine locale
spark.conf.set("spark.sql.shuffle.partitions", "12")

# 2. Définition de la requête (Idem Baseline)
df_silver = spark.read.parquet(CFG["paths"]["silver"])

gold_q1_opt = (df_silver
           .groupBy("curr")
           .agg(F.sum("n").alias("total_clicks"))
           .orderBy(F.col("total_clicks").desc())
          )

# 3. Mesure du temps (Version Optimisée)
start_time = time.time()

gold_path_q1_opt = f"{CFG['paths']['gold']}/q1_optimized"

# OPTIMISATION D'ÉCRITURE :
# .coalesce(1) : On rassemble tout en 1 seul fichier propre (idéal pour le reporting final)
# ou on garde le partitionnement naturel réduit.
# Ici, on écrit direct. Le gain viendra du paramètre shuffle réglé plus haut.
gold_q1_opt.write.mode("overwrite").parquet(gold_path_q1_opt)

end_time = time.time()
duration_opt = round(end_time - start_time, 2)

# 4. Sauvegarde de la preuve
plan_opt = gold_q1_opt._jdf.queryExecution().executedPlan().toString()
with open(f"{CFG['paths']['proof']}/optimized_q1_plan.txt", "w") as f:
    f.write(f"Timestamp: {datetime.datetime.now()}\n")
    f.write(f"Duration: {duration_opt}s\n")
    f.write("Configuration: spark.sql.shuffle.partitions = 12\n")
    f.write("Plan Optimisé Q1:\n")
    f.write(plan_opt)

print("-" * 30)
print(f"⏱️ Temps Baseline : 9.44 s")
print(f"🚀 Temps Optimisé : {duration_opt} s")

print(f"📂 Données écrites dans : {gold_path_q1_opt}")

# Remettre la config par défaut pour ne pas fausser d'autres tests futurs
spark.conf.set("spark.sql.shuffle.partitions", "200")

⚡ Lancement de la Gold Layer - Q1 OPTIMIZED...


{"ts":"2026-01-03T17:33:22.062Z","level":"INFO","msg":"It took 4 ms to list leaf files for 1 paths.","logger":"InMemoryFileIndex"}
{"ts":"2026-01-03T17:33:22.091Z","level":"INFO","msg":"Starting job: parquet at NativeMethodAccessorImpl.java:0","logger":"SparkContext"}
{"ts":"2026-01-03T17:33:22.092Z","level":"INFO","msg":"Got job 26 (parquet at NativeMethodAccessorImpl.java:0) with 1 output partitions","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:33:22.092Z","level":"INFO","msg":"Final stage: ResultStage 35 (parquet at NativeMethodAccessorImpl.java:0)","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:33:22.092Z","level":"INFO","msg":"Parents of final stage: List()","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:33:22.092Z","level":"INFO","msg":"Missing parents: List()","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:33:22.093Z","level":"INFO","msg":"Submitting ResultStage 35 (MapPartitionsRDD[97] at parquet at NativeMethodAccessorImpl.java:0), which has no missing parents","logger":"DAGSche

------------------------------
⏱️ Temps Baseline : 9.44 s
🚀 Temps Optimisé : 5.38 s
📂 Données écrites dans : /home/zettahorizon/projects/de1_wikipedia/data/gold/q1_optimized


{"ts":"2026-01-03T17:33:27.565Z","level":"INFO","msg":"Job 30 is finished. Cancelling potential speculative or zombie tasks for this job","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:33:27.566Z","level":"INFO","msg":"Canceling stage 43","logger":"TaskSchedulerImpl"}
{"ts":"2026-01-03T17:33:27.566Z","level":"INFO","msg":"Killing all running tasks in stage 43: Stage finished","logger":"TaskSchedulerImpl"}
{"ts":"2026-01-03T17:33:27.566Z","level":"INFO","msg":"Job 30 finished: parquet at NativeMethodAccessorImpl.java:0, took 0.572094 s","logger":"DAGScheduler"}
{"ts":"2026-01-03T17:33:27.567Z","level":"INFO","msg":"Start to commit write Job 52d8d28e-511a-447e-8cac-771ee8339d6b.","logger":"FileFormatWriter"}
{"ts":"2026-01-03T17:33:27.582Z","level":"INFO","msg":"Write Job 52d8d28e-511a-447e-8cac-771ee8339d6b committed. Elapsed time: 15 ms.","logger":"FileFormatWriter"}
{"ts":"2026-01-03T17:33:27.583Z","level":"INFO","msg":"Finished processing stats for write job 52d8d28e-511a-447e-8cac-77

Comparaison temps

In [16]:
import datetime

metrics_path = CFG["paths"]["metrics"]
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# On note les deux runs
lines_to_append = [
    f"run_1,{timestamp},gold,q1_baseline_aggregation,9.44,0,0,Shuffle Default (200)",
    f"run_1,{timestamp},gold,q1_optimized_aggregation,5.38,0,0,Shuffle Tuned (12)"
]

with open(metrics_path, "a") as f:
    for line in lines_to_append:
        f.write(line + "\n")

print(f"✅ Scores enregistrés dans : {metrics_path}")

✅ Scores enregistrés dans : /home/zettahorizon/projects/de1_wikipedia/project_metrics_log.csv


In [18]:
# Cellule de vérification des métriques
import pandas as pd # Si pandas est installé
# Sinon, on utilise une lecture standard :

print("📋 Contenu du Journal des Métriques :")
print("-" * 60)
with open(CFG["paths"]["metrics"], "r") as f:
    print(f.read())
print("-" * 60)

📋 Contenu du Journal des Métriques :
------------------------------------------------------------
run_1,2026-01-03 18:28:02,bronze,ingest_csv_to_parquet,64.1,0,35605767,Raw TSV to Bronze
run_1,2026-01-03 18:28:02,silver,clean_and_cast,10.9,35605767,35605763,Cast n to Int & DropNulls
run_1,2026-01-03 18:35:10,gold,q1_baseline_aggregation,9.44,0,0,Shuffle Default (200)
run_1,2026-01-03 18:35:10,gold,q1_optimized_aggregation,5.38,0,0,Shuffle Tuned (12)

------------------------------------------------------------


Arrêt de la session

In [19]:
spark.stop()
print("🛑 Session Spark terminée. Projet technique validé !")

{"ts":"2026-01-03T17:45:48.701Z","level":"INFO","msg":"SparkContext is stopping with exitCode 0 from stop at NativeMethodAccessorImpl.java:0.","logger":"SparkContext"}
{"ts":"2026-01-03T17:45:48.723Z","level":"INFO","msg":"Stopped Spark web UI at http://10.255.255.254:4040","logger":"SparkUI"}
{"ts":"2026-01-03T17:45:48.742Z","level":"INFO","msg":"MapOutputTrackerMasterEndpoint stopped!","logger":"MapOutputTrackerMasterEndpoint"}
{"ts":"2026-01-03T17:45:48.796Z","level":"INFO","msg":"MemoryStore cleared","logger":"MemoryStore"}
{"ts":"2026-01-03T17:45:48.797Z","level":"INFO","msg":"BlockManager stopped","logger":"BlockManager"}
{"ts":"2026-01-03T17:45:48.801Z","level":"INFO","msg":"BlockManagerMaster stopped","logger":"BlockManagerMaster"}
{"ts":"2026-01-03T17:45:48.806Z","level":"INFO","msg":"OutputCommitCoordinator stopped!","logger":"OutputCommitCoordinator$OutputCommitCoordinatorEndpoint"}
{"ts":"2026-01-03T17:45:48.819Z","level":"INFO","msg":"Successfully stopped SparkContext","lo

🛑 Session Spark terminée. Projet technique validé !
